# Basic Queries

## 1:Import required libraries

In [1]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime

### 2:Database creation and connection 

In [2]:
import sqlite3
conn = sqlite3.connect("retail.db")
cursor=conn.cursor()
print("Database Connected")

Database Connected


## 3:Table Creation

In [3]:
customers = pd.read_csv("customers.csv")
products = pd.read_csv("cleaned_products.csv")
orders = pd.read_csv("cleaned_orders.csv")
order_items = pd.read_csv("order_items.csv")

In [4]:
customers.to_sql("customers",conn,if_exists="replace",index=False)

products.to_sql("products",conn,if_exists="replace",index=False)

orders.to_sql("orders",conn,if_exists="replace",index=False)

order_items.to_sql("order_items",conn,if_exists="replace",index=False)

500

## 4:Query run function

In [5]:
def run_query(query):
    return pd.read_sql_query(query,conn)

##### 1. Total revenue per category (revenue = quantity × unit_price × (1 - discount_percent/100))


In [6]:
q1="""select category ,sum((quantity*unit_price*(1 - discount_price/100))) as revenue from products p
inner join order_items oi
on p.product_id=oi.product_id 
group by category
"""
df_q1=run_query(q1)
df_q1

,Category,revenue
0,Books,1267012
1,Clothing,1428035
2,Electronics,1518952
3,Home,1391033


##### 2. Top 10 customers by total order value


In [7]:
q2="""
select c.customer_id,c.customer_name,round(sum(oi.quantity *oi.unit_price *(1 - oi.discount_price / 100.0)),2) as total_order_value
from customers c
join orders o
on c.customer_id = o.customer_id
join order_items oi
on o.order_id = oi.order_id
group by c.customer_id,c.customer_name
order by total_order_value desc
limit 10;
"""
df_q2=run_query(q2)
df_q2

,customer_id,customer_name,total_order_value
0,343,Anthony Vega,73498.55
1,245,Dennis Wiley IV,71957.44
2,391,Sandra Hatfield,58512.32
3,37,William Cooper,48443.28
4,41,Cassidy Long,47620.94
5,237,Michelle Faulkner MD,42346.56
6,400,Cheryl Thompson,42065.23
7,71,Linda Stephenson,40224.10
8,388,Brenda Ballard,38045.14
9,405,Tammy Rosales,37747.83


##### 3. Month-wise order count for the last 12 months


In [9]:
q3="""
select strftime('%Y-%m', order_date) as month,
count(*) as total_orders
from orders
group by strftime('%Y-%m', order_date)
order by month desc
limit 12"""
df_q3=run_query(q3)
df_q3

,month,total_orders
0,2026-07,5
1,2026-06,17
2,2026-05,22
3,2026-04,23
4,2026-03,19
5,2026-02,16
6,2026-01,21
7,2025-12,23
8,2025-11,17
9,2025-10,25


## 4:Intermediate queries

##### 4. Find customers who placed orders but never had any item delivered


In [11]:
q4=""" select distinct c.customer_id,c.customer_name
from customers c
left join orders o
on c.customer_id = o.customer_id
and o.status = 'DELIVERED'
where o.order_id is null"""
df_q4=run_query(q4)
df_q4

,customer_id,customer_name
0,1,Jacob Walker
1,2,Marie Hill
2,3,Brittany Williams
3,5,Kent Dean
4,9,Katherine Murray
...,...,...
403,492,Mr. Scott Nichols Jr.
404,494,Kelly Wallace
405,498,Harold Roberts
406,499,William Medina MD


##### 5. Products that were ordered but had more returns than purchases


In [13]:
q5="""select p.product_id,p.product_name,sum(case when o.status = 'RETURNED' then 1 else 0 end) as returns,
sum(case when o.status = 'DELIVERED' then 1 else 0 end) as purchases
from products p
inner join order_items oi
on p.product_id = oi.product_id
join orders o
on oi.order_id = o.order_id
group by p.product_id,p.product_name
having returns > purchases;"""
df_q5=run_query(q5)
df_q5

,Product_id,Product_name,returns,purchases
0,5,Bed,1,0
1,12,Sony24,1,0
2,13,Nikon300,1,0
3,14,Ideapad,1,0
4,32,Lower,2,0
...,...,...,...,...
71,468,Mice,1,0
72,472,Macbook,1,0
73,478,Bed,1,0
74,484,Table,1,0


##### 6. Calculate the return rate (returned items / total items) per category


In [22]:
q6="""select p.category,sum(case when o.status='RETURNED'then oi.quantity
else 0
end) as returned_items,
sum(oi.quantity) as total_items,round(
sum(case when o.status='RETURNED'then oi.quantity
else 0 end)*100.0/sum(oi.quantity),2) as return_rate
from products p
join order_items oi
on p.product_id=oi.product_id
join orders o
on oi.order_id=o.order_id
group by p.category"""
df_q6=run_query(q6)
df_q6

,Category,returned_items,total_items,return_rate
0,Books,59,332,17.77
1,Clothing,82,337,24.33
2,Electronics,110,401,27.43
3,Home,66,368,17.93
